# AED v6 — YAMNet + MFCC, meta de 90%

Esta versão muda a estratégia: em vez de treinar outro classificador
visual de espectrogramas, usa transferência de aprendizado com YAMNet,
pré-treinado em 521 eventos acústicos, e combina seus embeddings com
estatísticas de MFCC.

O notebook amplia os dados com:

- ASVP-ESD;
- Scream/Non-Scream;
- ESC-50;
- UrbanSound8K;
- TUT Rare Sound Events;
- Gunshot Audio Dataset;
- Dataset-AOB;
- TAU Sound Events and Speech Privacy Preservation.

Classes:

1. `ambiente_normal`
2. `grito_panico`
3. `choro_distress`
4. `impacto`
5. `vidro_quebrando`
6. `tiro`

O modelo só marca a meta como atingida quando, no teste por grupos
independentes:

- acurácia `>= 90%`;
- F1 macro `>= 90%`;
- nenhuma classe tem F1 abaixo de `80%`.

A meta é um critério de aceitação, não uma garantia.

In [ ]:
!apt-get -qq update
!apt-get -qq install -y ffmpeg
!pip -q install             kagglehub             librosa==0.11.0             soundfile             scikit-learn             pandas             matplotlib             tqdm             pyyaml             requests             tensorflow-hub

In [ ]:
from collections import Counter
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Any

import gc
import hashlib
import json
import math
import random
import re
import shutil
import zipfile

import kagglehub
import librosa
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
import soundfile as sf
import tensorflow as tf
import tensorflow_hub as hub

from google.colab import files
from IPython.display import Audio, display
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.preprocessing import StandardScaler
from tqdm.auto import tqdm

print("TensorFlow:", tf.__version__)
print("GPU:", tf.config.list_physical_devices("GPU"))

if not tf.config.list_physical_devices("GPU"):
    raise RuntimeError(
        "Ative uma GPU T4 no Colab."
    )

for gpu in tf.config.list_physical_devices("GPU"):
    try:
        tf.config.experimental.set_memory_growth(
            gpu,
            True,
        )
    except RuntimeError:
        pass

tf.keras.mixed_precision.set_global_policy(
    "mixed_float16"
)

## 1. Configuração

In [ ]:
@dataclass(frozen=True)
class Configuracao:
    sr: int = 16000
    duracao: float = 2.0
    passo: float = 0.50

    n_mfcc: int = 40
    n_fft: int = 512
    hop_length: int = 256

    max_arquivos_por_classe: int = 5000
    max_janelas_por_arquivo: int = 4
    augmentacoes_raras: int = 2

    lote: int = 64
    epocas: int = 80
    paciencia: int = 12
    seed: int = 42

    meta_accuracy: float = 0.90
    meta_f1_macro: float = 0.90
    meta_f1_classe: float = 0.80

    usar_aob: bool = True
    usar_tau_privacy: bool = True
    usar_tut: bool = True
    usar_reforco_manual: bool = False


CFG = Configuracao()

CLASSES = [
    "ambiente_normal",
    "grito_panico",
    "choro_distress",
    "impacto",
    "vidro_quebrando",
    "tiro",
]

ID = {
    classe: indice
    for indice, classe
    in enumerate(CLASSES)
}

FAMILIA = {
    "ambiente_normal": 0,
    "grito_panico": 1,
    "choro_distress": 1,
    "impacto": 2,
    "vidro_quebrando": 2,
    "tiro": 2,
}

RARAS = {
    "impacto",
    "vidro_quebrando",
    "tiro",
}

RAIZ = Path("/content/aed_v6_meta90")
DADOS = RAIZ / "datasets"
CACHE = RAIZ / "cache"
MODELOS = RAIZ / "modelos"
RESULTADOS = RAIZ / "resultados"
ENTREGA = RAIZ / "entrega"

ESC_DIR = DADOS / "esc50"
TUT_DIR = DADOS / "tut_rare"
AOB_DEV = DADOS / "aob_dev"
AOB_EVAL = DADOS / "aob_eval"
TAU_DIR = DADOS / "tau_privacy"
REFORCO = DADOS / "reforco_manual"

for pasta in (
    RAIZ,
    DADOS,
    CACHE,
    MODELOS,
    RESULTADOS,
    ENTREGA,
):
    pasta.mkdir(
        parents=True,
        exist_ok=True,
    )

random.seed(CFG.seed)
np.random.seed(CFG.seed)
tf.random.set_seed(CFG.seed)

display(pd.DataFrame([asdict(CFG)]))

## 2. Download dos datasets

In [ ]:
ASVP = Path(
    kagglehub.dataset_download(
        (
            "dejolilandry/"
            "asvpesdspeech-nonspeech-emotional-utterances"
        )
    )
)

GUNSHOT = Path(
    kagglehub.dataset_download(
        "emrahaydemr/gunshot-audio-dataset"
    )
)

URBAN = Path(
    kagglehub.dataset_download(
        "chrisfilo/urbansound8k"
    )
)

SCREAM = Path(
    kagglehub.dataset_download(
        (
            "aananehsansiam/"
            "audio-dataset-of-scream-and-non-scream"
        )
    )
)

print("ASVP:", ASVP)
print("Gunshot:", GUNSHOT)
print("UrbanSound8K:", URBAN)
print("Scream:", SCREAM)

In [ ]:
def zip_valido(
    caminho: Path,
) -> bool:
    return (
        caminho.exists()
        and caminho.stat().st_size > 0
        and zipfile.is_zipfile(caminho)
    )


def baixar_zip(
    url: str,
    destino: Path,
) -> None:
    if zip_valido(destino):
        print("Já disponível:", destino)
        return

    destino.unlink(missing_ok=True)
    temporario = destino.with_suffix(
        destino.suffix + ".part"
    )
    temporario.unlink(missing_ok=True)

    ultimo_erro = None

    for verificar_ssl in (True, False):
        try:
            print(
                "Baixando:",
                url,
                "| SSL:",
                verificar_ssl,
            )

            with requests.get(
                url,
                stream=True,
                timeout=(30, 240),
                verify=verificar_ssl,
                allow_redirects=True,
                headers={
                    "User-Agent": (
                        "Mozilla/5.0 "
                        "violence-edge-ai-aed-v6"
                    )
                },
            ) as resposta:
                resposta.raise_for_status()

                total = int(
                    resposta.headers.get(
                        "content-length",
                        0,
                    )
                )

                with temporario.open("wb") as arquivo:
                    with tqdm(
                        total=total,
                        unit="B",
                        unit_scale=True,
                        desc=destino.name,
                    ) as barra:
                        for bloco in resposta.iter_content(
                            chunk_size=1024 * 1024
                        ):
                            if not bloco:
                                continue

                            arquivo.write(bloco)
                            barra.update(len(bloco))

            temporario.replace(destino)

            if not zip_valido(destino):
                destino.unlink(missing_ok=True)
                raise RuntimeError(
                    "O arquivo não é um ZIP válido."
                )

            return

        except Exception as erro:
            ultimo_erro = erro
            temporario.unlink(missing_ok=True)
            print(
                "Tentativa falhou:",
                type(erro).__name__,
                erro,
            )

    raise RuntimeError(
        "Falha no download. "
        f"Último erro: {ultimo_erro}"
    )


def extrair_zip(
    arquivo_zip: Path,
    destino: Path,
) -> None:
    marcador = destino / ".extraido"

    if marcador.exists():
        return

    destino.mkdir(
        parents=True,
        exist_ok=True,
    )

    with zipfile.ZipFile(
        arquivo_zip
    ) as arquivo:
        arquivo.extractall(destino)

    marcador.touch()


ZIP_ESC = DADOS / "esc50.zip"

baixar_zip(
    (
        "https://github.com/karolpiczak/"
        "ESC-50/archive/refs/heads/master.zip"
    ),
    ZIP_ESC,
)

extrair_zip(
    ZIP_ESC,
    ESC_DIR,
)

if CFG.usar_tut:
    ZIP_TUT = DADOS / "tut_rare.zip"

    baixar_zip(
        (
            "https://zenodo.org/records/401395/files/"
            "TUT-rare-sound-events-2017-development."
            "source_data_events.zip?download=1"
        ),
        ZIP_TUT,
    )

    extrair_zip(
        ZIP_TUT,
        TUT_DIR,
    )

if CFG.usar_aob:
    ZIP_AOB_DEV = DADOS / "aob_dev.zip"
    ZIP_AOB_EVAL = DADOS / "aob_eval.zip"

    baixar_zip(
        (
            "https://zenodo.org/records/4319802/files/"
            "DATASET_AOB.zip?download=1"
        ),
        ZIP_AOB_DEV,
    )

    baixar_zip(
        (
            "https://zenodo.org/records/4319802/files/"
            "EVALUATION_dataset_AOB.zip?download=1"
        ),
        ZIP_AOB_EVAL,
    )

    extrair_zip(
        ZIP_AOB_DEV,
        AOB_DEV,
    )

    extrair_zip(
        ZIP_AOB_EVAL,
        AOB_EVAL,
    )

if CFG.usar_tau_privacy:
    ZIP_TAU = DADOS / "tau_privacy.zip"

    baixar_zip(
        (
            "https://zenodo.org/records/10378145/files/"
            "final_data_wav_split.zip?download=1"
        ),
        ZIP_TAU,
    )

    extrair_zip(
        ZIP_TAU,
        TAU_DIR,
    )

print("Downloads concluídos.")

## 3. Reforço manual opcional

In [ ]:
if CFG.usar_reforco_manual:
    enviados = files.upload()

    zips = [
        nome
        for nome in enviados
        if nome.lower().endswith(".zip")
    ]

    if len(zips) != 1:
        raise RuntimeError(
            "Envie exatamente um ZIP."
        )

    if REFORCO.exists():
        shutil.rmtree(REFORCO)

    REFORCO.mkdir(
        parents=True,
        exist_ok=True,
    )

    with zipfile.ZipFile(zips[0]) as arquivo:
        arquivo.extractall(REFORCO)

    print(
        "Reforço:",
        REFORCO,
    )
else:
    print("Reforço manual desativado.")

## 4. Metadados unificados

In [ ]:
EXTENSOES = {
    ".wav",
    ".mp3",
    ".ogg",
    ".flac",
    ".m4a",
}


def audios_em(
    pasta: Path | None,
) -> list[Path]:
    if pasta is None or not pasta.exists():
        return []

    return sorted(
        caminho
        for caminho in pasta.rglob("*")
        if (
            caminho.is_file()
            and caminho.suffix.lower()
            in EXTENSOES
        )
    )


def token(texto: str) -> str:
    return re.sub(
        r"[^a-z0-9]+",
        "_",
        str(texto).lower(),
    ).strip("_")


def adicionar(
    registros: list[dict[str, Any]],
    arquivo: Path,
    classe: str,
    fonte: str,
    grupo: str,
    split_fixo: str = "",
    extra: str = "",
) -> None:
    arquivo = Path(arquivo)

    if arquivo.exists() and classe in ID:
        registros.append(
            {
                "arquivo": str(arquivo.resolve()),
                "classe": classe,
                "classe_id": ID[classe],
                "familia": FAMILIA[classe],
                "perigo": int(
                    classe != "ambiente_normal"
                ),
                "fonte": fonte,
                "grupo": str(grupo),
                "split_fixo": split_fixo,
                "extra": str(extra),
            }
        )


registros = []


# ASVP
mapa_asvp = {
    "02": "ambiente_normal",
    "04": "choro_distress",
    "06": "grito_panico",
}

for arquivo in audios_em(ASVP):
    numeros = re.findall(
        r"\d+",
        arquivo.stem,
    )

    if len(numeros) < 6:
        continue

    emocao = numeros[2].zfill(2)
    ator = numeros[5].zfill(3)
    classe = mapa_asvp.get(emocao)

    if classe is None:
        continue

    adicionar(
        registros,
        arquivo,
        classe,
        "ASVP",
        f"asvp:ator_{ator}",
        extra=emocao,
    )


# Scream / non-scream
for arquivo in audios_em(SCREAM):
    relativo = arquivo.relative_to(
        SCREAM
    )

    pastas = [
        token(parte)
        for parte in relativo.parts[:-1]
    ]

    if any(
        nome in {
            "non_scream",
            "nonscream",
            "no_scream",
            "not_scream",
        }
        for nome in pastas
    ):
        classe = "ambiente_normal"

    elif any(
        nome in {
            "scream",
            "screams",
        }
        for nome in pastas
    ):
        classe = "grito_panico"

    else:
        continue

    adicionar(
        registros,
        arquivo,
        classe,
        "ScreamDataset",
        (
            f"scream:{classe}:"
            f"{token(arquivo.stem)}"
        ),
        extra="/".join(pastas),
    )


# Gunshot
for arquivo in audios_em(GUNSHOT):
    relativo = arquivo.relative_to(
        GUNSHOT
    )

    origem = token(
        relativo.parts[0]
        if relativo.parts
        else arquivo.parent.name
    )

    adicionar(
        registros,
        arquivo,
        "tiro",
        "GunshotKaggle",
        f"gunshot:{origem}",
        extra=origem,
    )


# ESC-50
csvs_esc = list(
    ESC_DIR.rglob("esc50.csv")
)
pastas_esc = [
    pasta
    for pasta in ESC_DIR.rglob("audio")
    if pasta.is_dir()
]

if not csvs_esc or not pastas_esc:
    raise RuntimeError(
        "ESC-50 incompleto."
    )

tabela_esc = pd.read_csv(
    csvs_esc[0]
)
audio_esc = pastas_esc[0]

mapa_esc = {
    "crying_baby": "choro_distress",
    "glass_breaking": "vidro_quebrando",
    "door_wood_knock": "impacto",
}

normais_esc = {
    "clapping",
    "hand_saw",
    "door_wood_creaks",
    "fireworks",
    "chainsaw",
    "siren",
    "engine",
    "vacuum_cleaner",
    "washing_machine",
    "clock_alarm",
    "can_opening",
    "mouse_click",
    "keyboard_typing",
    "footsteps",
    "laughing",
    "breathing",
    "coughing",
    "sneezing",
    "snoring",
    "rain",
    "sea_waves",
    "wind",
    "thunderstorm",
    "airplane",
    "helicopter",
    "train",
}

for linha in tabela_esc.itertuples():
    categoria = str(linha.category)

    if categoria in mapa_esc:
        classe = mapa_esc[categoria]
    elif categoria in normais_esc:
        classe = "ambiente_normal"
    else:
        continue

    adicionar(
        registros,
        audio_esc / str(linha.filename),
        classe,
        "ESC-50",
        f"esc50:{linha.src_file}",
        extra=categoria,
    )


# UrbanSound8K
csvs_urban = list(
    URBAN.rglob("UrbanSound8K.csv")
)

if csvs_urban:
    tabela_urban = pd.read_csv(
        csvs_urban[0]
    )
    raiz_audio = (
        csvs_urban[0].parent.parent
        / "audio"
    )

    for _, linha in tabela_urban.iterrows():
        nome = str(linha["class"])

        if nome == "gun_shot":
            classe = "tiro"
        elif nome in {
            "jackhammer",
            "drilling",
            "siren",
            "car_horn",
            "dog_bark",
            "street_music",
            "children_playing",
            "air_conditioner",
            "engine_idling",
        }:
            classe = "ambiente_normal"
        else:
            continue

        arquivo = (
            raiz_audio
            / f"fold{int(linha['fold'])}"
            / str(linha["slice_file_name"])
        )

        fsid = str(
            linha.get(
                "fsID",
                str(
                    linha["slice_file_name"]
                ).split("-")[0],
            )
        )

        adicionar(
            registros,
            arquivo,
            classe,
            "UrbanSound8K",
            f"urban:fsid_{fsid}",
            extra=nome,
        )


# TUT Rare
if CFG.usar_tut:
    for arquivo in audios_em(TUT_DIR):
        texto = token(str(arquivo))

        if any(
            termo in texto
            for termo in (
                "babycry",
                "baby_cry",
                "babycrying",
            )
        ):
            classe = "choro_distress"
        elif any(
            termo in texto
            for termo in (
                "glassbreak",
                "glass_break",
                "glassbreaking",
            )
        ):
            classe = "vidro_quebrando"
        elif any(
            termo in texto
            for termo in (
                "gunshot",
                "gun_shot",
                "gunfire",
            )
        ):
            classe = "tiro"
        else:
            continue

        adicionar(
            registros,
            arquivo,
            classe,
            "TUT_Rare",
            f"tut:{classe}:{arquivo.stem}",
            extra=classe,
        )


# Dataset-AOB
def classe_aob(
    caminho: Path,
) -> str | None:
    texto = token(str(caminho))

    if any(
        termo in texto
        for termo in (
            "glass_breaking",
            "glassbreaking",
            "glass_break",
        )
    ):
        return "vidro_quebrando"

    if any(
        termo in texto
        for termo in (
            "gun_shot",
            "gunshot",
        )
    ):
        return "tiro"

    if any(
        termo in texto
        for termo in (
            "screams",
            "scream",
        )
    ):
        return "grito_panico"

    if any(
        termo in texto
        for termo in (
            "alarm_siren",
            "children_playing",
            "dog_bark",
            "engine",
            "footsteps",
            "metro_train",
            "rain",
        )
    ):
        return "ambiente_normal"

    return None


if CFG.usar_aob:
    for raiz_aob, split_fixo in (
        (AOB_DEV, ""),
        (AOB_EVAL, "externo"),
    ):
        for arquivo in audios_em(
            raiz_aob
        ):
            classe = classe_aob(
                arquivo
            )

            if classe is None:
                continue

            relativo = arquivo.relative_to(
                raiz_aob
            )

            grupo_base = token(
                relativo.stem
            )

            adicionar(
                registros,
                arquivo,
                classe,
                "Dataset-AOB",
                (
                    f"aob:{split_fixo or 'dev'}:"
                    f"{grupo_base}"
                ),
                split_fixo=split_fixo,
                extra=str(relativo.parent),
            )


# TAU Privacy
def classe_tau(
    caminho: Path,
) -> str | None:
    texto = token(str(caminho))

    if "glass_breaking" in texto:
        return "vidro_quebrando"

    if (
        "gun_shot" in texto
        or "gunshot" in texto
    ):
        return "tiro"

    if "slam" in texto:
        return "impacto"

    if "crying" in texto:
        return "choro_distress"

    if any(
        termo in texto
        for termo in (
            "dog_barking",
            "cough",
            "applause",
            "dished_pot_pan",
            "toilet_flush",
            "cat_meowing",
            "doorbell",
            "drill",
        )
    ):
        return "ambiente_normal"

    return None


if CFG.usar_tau_privacy:
    for arquivo in audios_em(TAU_DIR):
        classe = classe_tau(arquivo)

        if classe is None:
            continue

        base = re.sub(
            r"(_speech.*|_nospeech.*)$",
            "",
            token(arquivo.stem),
        )

        adicionar(
            registros,
            arquivo,
            classe,
            "TAU-Privacy",
            f"tau:{base}",
            extra=str(
                arquivo.relative_to(
                    TAU_DIR
                ).parent
            ),
        )


# Reforço manual
if CFG.usar_reforco_manual:
    for classe in CLASSES:
        for pasta in REFORCO.rglob(
            classe
        ):
            if not pasta.is_dir():
                continue

            for arquivo in audios_em(
                pasta
            ):
                adicionar(
                    registros,
                    arquivo,
                    classe,
                    "ReforcoManual",
                    (
                        f"manual:{classe}:"
                        f"{token(arquivo.stem)}"
                    ),
                    extra=classe,
                )


meta = pd.DataFrame(registros)

if meta.empty:
    raise RuntimeError(
        "Nenhum áudio encontrado."
    )

meta = meta.drop_duplicates(
    subset=["arquivo", "classe"]
)

desenvolvimento = meta[
    meta["split_fixo"] != "externo"
].copy()

externo = meta[
    meta["split_fixo"] == "externo"
].copy()

partes = []

for classe in CLASSES:
    parte = desenvolvimento[
        desenvolvimento["classe"]
        == classe
    ].copy()

    if (
        len(parte)
        > CFG.max_arquivos_por_classe
    ):
        parte = parte.sample(
            CFG.max_arquivos_por_classe,
            random_state=(
                CFG.seed + ID[classe]
            ),
        )

    partes.append(parte)

desenvolvimento = pd.concat(
    partes,
    ignore_index=True,
)

display(
    desenvolvimento.groupby(
        ["classe", "fonte"]
    )
    .size()
    .unstack(fill_value=0)
    .reindex(
        CLASSES,
        fill_value=0,
    )
)

resumo = (
    desenvolvimento.groupby(
        "classe"
    )
    .agg(
        arquivos=("arquivo", "count"),
        grupos=("grupo", "nunique"),
    )
    .reindex(
        CLASSES,
        fill_value=0,
    )
)

display(resumo)

faltantes = [
    classe
    for classe in CLASSES
    if resumo.loc[
        classe,
        "arquivos",
    ] == 0
]

if faltantes:
    raise RuntimeError(
        f"Classes sem dados: {faltantes}"
    )

desenvolvimento.to_csv(
    RESULTADOS
    / "fontes_desenvolvimento.csv",
    index=False,
)

externo.to_csv(
    RESULTADOS
    / "fontes_externo_aob.csv",
    index=False,
)

## 5. Auditoria manual

In [ ]:
for classe in CLASSES:
    print("\n" + "=" * 72)
    print("Classe:", classe)

    parte = desenvolvimento[
        desenvolvimento["classe"]
        == classe
    ]

    for linha in (
        parte.sample(
            min(2, len(parte)),
            random_state=(
                CFG.seed + ID[classe]
            ),
        )
        .itertuples()
    ):
        print(
            linha.fonte,
            "|",
            Path(linha.arquivo).name,
            "|",
            linha.extra,
        )
        display(
            Audio(linha.arquivo)
        )

In [ ]:
APROVAR_AUDITORIA = True

if not APROVAR_AUDITORIA:
    raise RuntimeError(
        "Ouça as amostras e altere "
        "APROVAR_AUDITORIA para True."
    )

## 6. Split por grupos

In [ ]:
def dividir_grupos(
    grupos: list[str],
    semente: int,
):
    grupos = list(
        dict.fromkeys(grupos)
    )

    random.Random(
        semente
    ).shuffle(grupos)

    total = len(grupos)

    if total < 3:
        raise RuntimeError(
            "Cada classe precisa de "
            "pelo menos 3 grupos."
        )

    n_treino = max(
        1,
        int(round(total * 0.70)),
    )
    n_validacao = max(
        1,
        int(round(total * 0.15)),
    )

    if (
        n_treino + n_validacao
        >= total
    ):
        n_treino = total - 2
        n_validacao = 1

    corte = n_treino + n_validacao

    return (
        set(grupos[:n_treino]),
        set(grupos[n_treino:corte]),
        set(grupos[corte:]),
    )


# Get all unique groups from the entire development dataset
all_unique_groups = desenvolvimento["grupo"].drop_duplicates().tolist()

# Split these unique groups into train, validation, and test sets ONCE
# Use a consistent seed for reproducibility
treino_grupos, validacao_grupos, teste_grupos = dividir_grupos(all_unique_groups, CFG.seed)

# Create a mapping from group name to split type
grupo_para_split = {}
for grupo in treino_grupos:
    grupo_para_split[grupo] = "treino"
for grupo in validacao_grupos:
    grupo_para_split[grupo] = "validacao"
for grupo in teste_grupos:
    grupo_para_split[grupo] = "teste"

# Assign the split type to each row in the desenvolvimento DataFrame
meta_split = desenvolvimento.copy()
meta_split["split"] = meta_split["grupo"].map(grupo_para_split)

# Ensure no group ended up with a NaN split (shouldn't happen if all_unique_groups covers everything)
if meta_split["split"].isnull().any():
    raise ValueError("Alguns grupos não foram atribuídos a um split.")

vazamento = (
    meta_split.groupby(
        "grupo"
    )["split"]
    .nunique()
    .gt(1)
    .sum()
)

if vazamento:
    raise RuntimeError(
        "Vazamento entre splits."
    )

display(
    meta_split.groupby(
        ["classe", "split"]
    )["grupo"]
    .nunique()
    .unstack(fill_value=0)
    .reindex(
        CLASSES,
        fill_value=0,
    )
)

meta_split.to_csv(
    RESULTADOS
    / "fontes_com_split.csv",
    index=False,
)

## 7. Extrair YAMNet + MFCC

In [ ]:
YAMNET_HANDLE = (
    "https://tfhub.dev/google/yamnet/1"
)

yamnet = hub.load(
    YAMNET_HANDLE
)

AMOSTRAS = int(
    CFG.sr * CFG.duracao
)


def carregar_audio(
    caminho: str | Path,
) -> np.ndarray:
    audio, _ = librosa.load(
        str(caminho),
        sr=CFG.sr,
        mono=True,
    )

    audio = np.nan_to_num(
        audio.astype(np.float32)
    )

    pico = float(
        np.max(np.abs(audio))
    )

    if pico > 1.0:
        audio = audio / pico

    return audio


def ajustar(
    audio: np.ndarray,
) -> np.ndarray:
    if len(audio) < AMOSTRAS:
        return np.pad(
            audio,
            (
                0,
                AMOSTRAS - len(audio),
            ),
        )

    return audio[:AMOSTRAS]


def janelas_uniformes(
    audio: np.ndarray,
) -> list[np.ndarray]:
    if len(audio) <= AMOSTRAS:
        return [ajustar(audio)]

    inicios = np.linspace(
        0,
        len(audio) - AMOSTRAS,
        CFG.max_janelas_por_arquivo,
        dtype=int,
    )

    return [
        audio[
            inicio:
            inicio + AMOSTRAS
        ]
        for inicio in inicios
    ]


def janelas_onset(
    audio: np.ndarray,
) -> list[np.ndarray]:
    if len(audio) <= AMOSTRAS:
        return [ajustar(audio)]

    onset = librosa.onset.onset_strength(
        y=audio,
        sr=CFG.sr,
        hop_length=CFG.hop_length,
    )

    picos = librosa.util.peak_pick(
        onset,
        pre_max=3,
        post_max=3,
        pre_avg=5,
        post_avg=5,
        delta=0.20,
        wait=8,
    )

    if len(picos) == 0:
        return janelas_uniformes(audio)

    ordem = np.argsort(
        onset[picos]
    )[::-1]

    janelas = []
    usados = []

    for indice in ordem:
        centro = librosa.frames_to_samples(
            int(picos[indice]),
            hop_length=CFG.hop_length,
        )

        inicio = int(
            np.clip(
                centro - AMOSTRAS // 2,
                0,
                len(audio) - AMOSTRAS,
            )
        )

        if any(
            abs(inicio - anterior)
            < CFG.sr * 0.30
            for anterior in usados
        ):
            continue

        janelas.append(
            audio[
                inicio:
                inicio + AMOSTRAS
            ]
        )
        usados.append(inicio)

        if (
            len(janelas)
            >= CFG.max_janelas_por_arquivo
        ):
            break

    return (
        janelas
        if janelas
        else janelas_uniformes(audio)
    )


def augmentar_waveform(
    audio: np.ndarray,
    rng: np.random.Generator,
) -> np.ndarray:
    audio = audio.copy()

    ganho_db = rng.uniform(
        -6.0,
        5.0,
    )

    audio *= 10 ** (
        ganho_db / 20.0
    )

    deslocamento = int(
        rng.uniform(
            -0.15,
            0.15,
        )
        * CFG.sr
    )

    audio = np.roll(
        audio,
        deslocamento,
    )

    ruido = rng.normal(
        0.0,
        rng.uniform(
            0.001,
            0.012,
        ),
        size=audio.shape,
    ).astype(np.float32)

    audio = audio + ruido

    return np.clip(
        audio,
        -1.0,
        1.0,
    )


def extrair_vetor(
    audio: np.ndarray,
) -> np.ndarray:
    audio = ajustar(audio).astype(
        np.float32
    )

    _, embeddings, _ = yamnet(
        tf.convert_to_tensor(
            audio,
            dtype=tf.float32,
        )
    )

    emb = embeddings.numpy()

    yamnet_stats = np.concatenate(
        [
            emb.mean(axis=0),
            emb.std(axis=0),
            emb.max(axis=0),
        ]
    )

    mfcc = librosa.feature.mfcc(
        y=audio,
        sr=CFG.sr,
        n_mfcc=CFG.n_mfcc,
        n_fft=CFG.n_fft,
        hop_length=CFG.hop_length,
    )

    delta = librosa.feature.delta(
        mfcc,
        width=9,
    )

    delta2 = librosa.feature.delta(
        mfcc,
        order=2,
        width=9,
    )

    mfcc_stats = np.concatenate(
        [
            mfcc.mean(axis=1),
            mfcc.std(axis=1),
            delta.mean(axis=1),
            delta.std(axis=1),
            delta2.mean(axis=1),
            delta2.std(axis=1),
        ]
    )

    espectral = np.asarray(
        [
            np.mean(
                librosa.feature.rms(
                    y=audio
                )
            ),
            np.std(
                librosa.feature.rms(
                    y=audio
                )
            ),
            np.mean(
                librosa.feature.spectral_centroid(
                    y=audio,
                    sr=CFG.sr,
                )
            ),
            np.std(
                librosa.feature.spectral_centroid(
                    y=audio,
                    sr=CFG.sr,
                )
            ),
            np.mean(
                librosa.feature.zero_crossing_rate(
                    audio
                )
            ),
            np.std(
                librosa.feature.zero_crossing_rate(
                    audio
                )
            ),
            np.max(np.abs(audio)),
        ],
        dtype=np.float32,
    )

    return np.concatenate(
        [
            yamnet_stats,
            mfcc_stats,
            espectral,
        ]
    ).astype(np.float32)


registros_features = []
vetores = []

rng = np.random.default_rng(
    CFG.seed
)

for linha in tqdm(
    meta_split.itertuples(),
    total=len(meta_split),
    desc="Extraindo features",
):
    try:
        audio = carregar_audio(
            linha.arquivo
        )

        if linha.classe in RARAS:
            janelas = janelas_onset(
                audio
            )
        else:
            janelas = janelas_uniformes(
                audio
            )

    except Exception as erro:
        print(
            "Falha:",
            linha.arquivo,
            erro,
        )
        continue

    for indice, janela in enumerate(
        janelas
    ):
        vetores.append(
            extrair_vetor(janela)
        )

        registros_features.append(
            {
                "classe": linha.classe,
                "classe_id": linha.classe_id,
                "familia": linha.familia,
                "perigo": linha.perigo,
                "grupo": linha.grupo,
                "split": linha.split,
                "fonte": linha.fonte,
                "arquivo": linha.arquivo,
                "augmentado": False,
            }
        )

        if (
            linha.split == "treino"
            and linha.classe in RARAS
        ):
            for _ in range(
                CFG.augmentacoes_raras
            ):
                aumentado = augmentar_waveform(
                    janela,
                    rng,
                )

                vetores.append(
                    extrair_vetor(
                        aumentado
                    )
                )

                registros_features.append(
                    {
                        "classe": linha.classe,
                        "classe_id": linha.classe_id,
                        "familia": linha.familia,
                        "perigo": linha.perigo,
                        "grupo": linha.grupo,
                        "split": linha.split,
                        "fonte": linha.fonte,
                        "arquivo": linha.arquivo,
                        "augmentado": True,
                    }
                )

X = np.asarray(
    vetores,
    dtype=np.float32,
)

tabela_features = pd.DataFrame(
    registros_features
)

np.save(
    CACHE / "features.npy",
    X,
)

tabela_features.to_csv(
    CACHE / "features.csv",
    index=False,
)

print(
    "Features:",
    X.shape,
)

display(
    tabela_features.groupby(
        ["classe", "split"]
    )
    .size()
    .unstack(fill_value=0)
    .reindex(
        CLASSES,
        fill_value=0,
    )
)

## 8. Normalização e datasets

In [ ]:
indices_treino = np.where(
    tabela_features["split"]
    .to_numpy()
    == "treino"
)[0]

indices_validacao = np.where(
    tabela_features["split"]
    .to_numpy()
    == "validacao"
)[0]

indices_teste = np.where(
    tabela_features["split"]
    .to_numpy()
    == "teste"
)[0]

scaler = StandardScaler()

X_treino = scaler.fit_transform(
    X[indices_treino]
).astype(np.float32)

X_validacao = scaler.transform(
    X[indices_validacao]
).astype(np.float32)

X_teste = scaler.transform(
    X[indices_teste]
).astype(np.float32)

np.savez_compressed(
    MODELOS
    / "normalizacao.npz",
    media=scaler.mean_.astype(
        np.float32
    ),
    escala=scaler.scale_.astype(
        np.float32
    ),
)

def rotulos(indices):
    parte = tabela_features.iloc[
        indices
    ]

    return {
        "evento": parte[
            "classe_id"
        ].astype(np.int32).to_numpy(),
        "familia": parte[
            "familia"
        ].astype(np.int32).to_numpy(),
        "perigo": parte[
            "perigo"
        ].astype(np.float32).to_numpy(),
    }

y_treino = rotulos(
    indices_treino
)
y_validacao = rotulos(
    indices_validacao
)
y_teste = rotulos(
    indices_teste
)

print(
    "Treino:",
    X_treino.shape,
)
print(
    "Validação:",
    X_validacao.shape,
)
print(
    "Teste:",
    X_teste.shape,
)

## 9. Modelo multirramo

In [ ]:
entrada = tf.keras.Input(
    shape=(X_treino.shape[1],),
    dtype=tf.float32,
    name="features",
)

x = tf.keras.layers.GaussianNoise(
    0.02
)(entrada)

x = tf.keras.layers.Dense(
    1024,
    activation="swish",
)(x)

x = tf.keras.layers.BatchNormalization()(x)
x = tf.keras.layers.Dropout(0.35)(x)

x = tf.keras.layers.Dense(
    512,
    activation="swish",
)(x)

x = tf.keras.layers.BatchNormalization()(x)
x = tf.keras.layers.Dropout(0.30)(x)

x = tf.keras.layers.Dense(
    256,
    activation="swish",
)(x)

x = tf.keras.layers.Dropout(0.20)(x)

evento = tf.keras.layers.Dense(
    len(CLASSES),
    activation="softmax",
    dtype=tf.float32,
    name="evento",
)(x)

familia = tf.keras.layers.Dense(
    3,
    activation="softmax",
    dtype=tf.float32,
    name="familia",
)(x)

perigo = tf.keras.layers.Dense(
    1,
    activation="sigmoid",
    dtype=tf.float32,
    name="perigo",
)(x)

modelo = tf.keras.Model(
    entrada,
    {
        "evento": evento,
        "familia": familia,
        "perigo": perigo,
    },
    name="aed_v6_yamnet_mfcc",
)

modelo.compile(
    optimizer=tf.keras.optimizers.AdamW(
        learning_rate=3e-4,
        weight_decay=1e-5,
    ),
    loss={
        "evento": (
            "sparse_categorical_crossentropy"
        ),
        "familia": (
            "sparse_categorical_crossentropy"
        ),
        "perigo": (
            "binary_crossentropy"
        ),
    },
    loss_weights={
        "evento": 1.0,
        "familia": 0.35,
        "perigo": 0.55,
    },
    metrics={
        "evento": [
            tf.keras.metrics.SparseCategoricalAccuracy(
                name="accuracy"
            )
        ],
        "familia": [
            tf.keras.metrics.SparseCategoricalAccuracy(
                name="accuracy"
            )
        ],
        "perigo": [
            tf.keras.metrics.BinaryAccuracy(
                name="accuracy"
            ),
            tf.keras.metrics.AUC(
                name="auc"
            ),
        ],
    },
)

modelo.summary()

## 10. Treinamento

In [ ]:
CHECKPOINT = (
    MODELOS
    / "aed_v6_melhor.keras"
)

callbacks = [
    tf.keras.callbacks.ModelCheckpoint(
        CHECKPOINT,
        monitor="val_evento_accuracy",
        mode="max",
        save_best_only=True,
        verbose=1,
    ),
    tf.keras.callbacks.EarlyStopping(
        monitor="val_evento_accuracy",
        mode="max",
        patience=CFG.paciencia,
        restore_best_weights=False,
        verbose=1,
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        mode="min",
        factor=0.5,
        patience=4,
        min_lr=1e-6,
        verbose=1,
    ),
    tf.keras.callbacks.CSVLogger(
        RESULTADOS
        / "historico_treinamento.csv"
    ),
]

historico = modelo.fit(
    X_treino,
    y_treino,
    validation_data=(
        X_validacao,
        y_validacao,
    ),
    epochs=CFG.epocas,
    batch_size=CFG.lote,
    callbacks=callbacks,
    shuffle=True,
    verbose=1,
)

modelo = tf.keras.models.load_model(
    CHECKPOINT
)

## 11. Avaliação e gate de 90%

In [ ]:
saidas = modelo.predict(
    X_teste,
    batch_size=CFG.lote,
    verbose=1,
)

probs_evento = saidas["evento"]
pred_evento = np.argmax(
    probs_evento,
    axis=1,
)

real_evento = y_teste[
    "evento"
]

accuracy = accuracy_score(
    real_evento,
    pred_evento,
)

balanced = (
    balanced_accuracy_score(
        real_evento,
        pred_evento,
    )
)

f1_macro = f1_score(
    real_evento,
    pred_evento,
    average="macro",
)

relatorio = pd.DataFrame(
    classification_report(
        real_evento,
        pred_evento,
        target_names=CLASSES,
        output_dict=True,
        digits=4,
    )
).T

display(relatorio)

f1_minimo = float(
    relatorio.loc[
        CLASSES,
        "f1-score",
    ].min()
)

atingiu_meta = bool(
    accuracy >= CFG.meta_accuracy
    and f1_macro
    >= CFG.meta_f1_macro
    and f1_minimo
    >= CFG.meta_f1_classe
)

resumo = {
    "accuracy": float(accuracy),
    "balanced_accuracy": float(
        balanced
    ),
    "f1_macro": float(
        f1_macro
    ),
    "menor_f1_classe": float(
        f1_minimo
    ),
    "atingiu_meta": (
        atingiu_meta
    ),
    "metas": {
        "accuracy": (
            CFG.meta_accuracy
        ),
        "f1_macro": (
            CFG.meta_f1_macro
        ),
        "f1_classe": (
            CFG.meta_f1_classe
        ),
    },
}

print(
    json.dumps(
        resumo,
        indent=2,
    )
)

relatorio.to_csv(
    RESULTADOS
    / "metricas_por_classe.csv"
)

(
    RESULTADOS
    / "resumo_metricas.json"
).write_text(
    json.dumps(
        resumo,
        ensure_ascii=False,
        indent=2,
    ),
    encoding="utf-8",
)

matriz = confusion_matrix(
    real_evento,
    pred_evento,
)

plt.figure(figsize=(10, 8))
plt.imshow(
    matriz,
    cmap="Blues",
)
plt.colorbar()
plt.title(
    "Matriz de confusão — AED v6"
)
plt.xticks(
    range(len(CLASSES)),
    CLASSES,
    rotation=45,
    ha="right",
)
plt.yticks(
    range(len(CLASSES)),
    CLASSES,
)

for linha in range(len(CLASSES)):
    for coluna in range(len(CLASSES)):
        plt.text(
            coluna,
            linha,
            str(matriz[linha, coluna]),
            ha="center",
            va="center",
        )

plt.xlabel("Previsto")
plt.ylabel("Real")
plt.tight_layout()

plt.savefig(
    RESULTADOS
    / "matriz_confusao.png",
    dpi=170,
)

plt.show()

## 12. Avaliação por gravação

In [ ]:
tabela_teste = (
    tabela_features.iloc[
        indices_teste
    ]
    .copy()
    .reset_index(drop=True)
)

for indice, classe in enumerate(
    CLASSES
):
    tabela_teste[
        f"prob_{classe}"
    ] = probs_evento[:, indice]

linhas = []

for grupo, parte in tabela_teste.groupby(
    "grupo"
):
    vetor = np.asarray(
        [
            parte[
                f"prob_{classe}"
            ].mean()
            for classe in CLASSES
        ]
    )

    real = int(
        parte["classe_id"]
        .mode()
        .iloc[0]
    )

    previsto = int(
        np.argmax(vetor)
    )

    linhas.append(
        {
            "grupo": grupo,
            "real": real,
            "previsto": previsto,
            "real_nome": CLASSES[real],
            "previsto_nome": (
                CLASSES[previsto]
            ),
            "confianca": float(
                vetor[previsto]
            ),
        }
    )

gravacoes = pd.DataFrame(
    linhas
)

metricas_gravacao = {
    "accuracy": float(
        accuracy_score(
            gravacoes["real"],
            gravacoes["previsto"],
        )
    ),
    "balanced_accuracy": float(
        balanced_accuracy_score(
            gravacoes["real"],
            gravacoes["previsto"],
        )
    ),
    "f1_macro": float(
        f1_score(
            gravacoes["real"],
            gravacoes["previsto"],
            average="macro",
        )
    ),
}

print(
    json.dumps(
        metricas_gravacao,
        indent=2,
    )
)

gravacoes.to_csv(
    RESULTADOS
    / "predicoes_por_gravacao.csv",
    index=False,
)

## 13. Avaliação externa Dataset-AOB

In [ ]:
if not externo.empty:
    vetores_externos = []
    linhas_externas = []

    for linha in tqdm(
        externo.itertuples(),
        total=len(externo),
        desc="AOB externo",
    ):
        try:
            audio = carregar_audio(
                linha.arquivo
            )

            janelas = (
                janelas_onset(audio)
                if linha.classe in RARAS
                else janelas_uniformes(
                    audio
                )
            )

            for janela in janelas:
                vetores_externos.append(
                    extrair_vetor(
                        janela
                    )
                )

                linhas_externas.append(
                    {
                        "classe": (
                            linha.classe
                        ),
                        "classe_id": (
                            linha.classe_id
                        ),
                        "grupo": (
                            linha.grupo
                        ),
                    }
                )

        except Exception as erro:
            print(
                "Falha externa:",
                linha.arquivo,
                erro,
            )

    if vetores_externos:
        X_externo = scaler.transform(
            np.asarray(
                vetores_externos,
                dtype=np.float32,
            )
        ).astype(np.float32)

        tabela_externa = pd.DataFrame(
            linhas_externas
        )

        saidas_externas = (
            modelo.predict(
                X_externo,
                batch_size=CFG.lote,
                verbose=1,
            )
        )

        pred_externo = np.argmax(
            saidas_externas["evento"],
            axis=1,
        )

        real_externo = tabela_externa[
            "classe_id"
        ].to_numpy()

        metricas_externas = {
            "accuracy": float(
                accuracy_score(
                    real_externo,
                    pred_externo,
                )
            ),
            "balanced_accuracy": float(
                balanced_accuracy_score(
                    real_externo,
                    pred_externo,
                )
            ),
            "f1_macro": float(
                f1_score(
                    real_externo,
                    pred_externo,
                    average="macro",
                )
            ),
        }

        print(
            json.dumps(
                metricas_externas,
                indent=2,
            )
        )

        (
            RESULTADOS
            / "metricas_externas_aob.json"
        ).write_text(
            json.dumps(
                metricas_externas,
                ensure_ascii=False,
                indent=2,
            ),
            encoding="utf-8",
        )
else:
    print(
        "Dataset-AOB externo não disponível."
    )

## 14. Exportar classificador TFLite

In [ ]:
TFLITE_FLOAT32 = (
    MODELOS
    / "aed_v6_classificador_float32.tflite"
)

TFLITE_FLOAT16 = (
    MODELOS
    / "aed_v6_classificador_float16.tflite"
)

conversor = (
    tf.lite.TFLiteConverter
    .from_keras_model(modelo)
)

# Adicionar suporte a operações Flex para o modelo Float32
conversor.target_spec.supported_ops = [
    tf.lite.OpsSet.TFLITE_BUILTINS,
    tf.lite.OpsSet.SELECT_TF_OPS
]

TFLITE_FLOAT32.write_bytes(
    conversor.convert()
)

conversor = (
    tf.lite.TFLiteConverter
    .from_keras_model(modelo)
)

conversor.optimizations = [
    tf.lite.Optimize.DEFAULT
]

conversor.target_spec.supported_types = [
    tf.float16
]

# Adicionar suporte a operações Flex para o modelo Float16
conversor.target_spec.supported_ops = [
    tf.lite.OpsSet.TFLITE_BUILTINS,
    tf.lite.OpsSet.SELECT_TF_OPS
]

TFLITE_FLOAT16.write_bytes(
    conversor.convert()
)

print(
    "Keras:",
    CHECKPOINT,
)
print(
    "TFLite Float32:",
    TFLITE_FLOAT32,
)
print(
    "TFLite Float16:",
    TFLITE_FLOAT16,
)

(
    MODELOS
    / "configuracao_aed_v6.json"
).write_text(
    json.dumps(
        {
            "classes": CLASSES,
            "configuracao": asdict(CFG),
            "yamnet_handle": (
                YAMNET_HANDLE
            ),
            "dimensao_features": int(
                X_treino.shape[1]
            ),
            "resultado": resumo,
            "resultado_gravacao": (
                metricas_gravacao
            ),
            "atingiu_meta": (
                atingiu_meta
            ),
        },
        ensure_ascii=False,
        indent=2,
    ),
    encoding="utf-8",
)

## 15. Pacote final

In [ ]:
if ENTREGA.exists():
    shutil.rmtree(ENTREGA)

ENTREGA.mkdir(
    parents=True,
    exist_ok=True,
)

for arquivo in MODELOS.glob("*"):
    if arquivo.is_file():
        shutil.copy2(
            arquivo,
            ENTREGA / arquivo.name,
        )

for arquivo in RESULTADOS.glob("*"):
    if arquivo.is_file():
        shutil.copy2(
            arquivo,
            ENTREGA / arquivo.name,
        )

ZIP_FINAL = shutil.make_archive(
    "/content/aed_v6_meta90_entrega",
    "zip",
    root_dir=str(ENTREGA),
)

print(
    "Meta atingida:",
    atingiu_meta,
)
print(
    "ZIP:",
    ZIP_FINAL,
)

files.download(
    ZIP_FINAL
)

## Observação de implantação

O classificador v6 usa embeddings do YAMNet e estatísticas de MFCC.
Na Raspberry Pi, a inferência terá dois estágios:

1. YAMNet para extrair embeddings da janela de áudio;
2. `aed_v6_classificador_float16.tflite` para classificar o evento.

O VAD continua separado e só aciona Whisper/NLP. Ele não bloqueia o AED.